# Logging Plots with MLflow

Plots tell you things tables don't — whether residuals are heteroscedastic, whether two features are collinear, whether the model misses on a specific slice. The question this notebook answers is: **where should those plots live so that, six weeks from now, you can still tell which run they belonged to?**

Saving them as PNGs in a `plots/` folder doesn't help — you lose the link to the params and metrics. MLflow's answer is `mlflow.log_figure(fig, "name.png")`: the figure is stored as a **run artifact**, side-by-side with the params, metrics, and model that produced it, and rendered inline in the UI's Artifacts tab.

This notebook is a modification of the official MLflow tutorial: **[Logging plots with MLflow](https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/part2-logging-plots/)**.  
Where this notebook corrects, supplements, or deliberately diverges from upstream, the relationship is named with an inline bold callout — one of **Bug**, **Stale**, **Missing from**, or **Diverges from upstream tutorial:** — or, in code cells, with a `NOTE (differs from upstream)` comment.  
Sections with no upstream counterpart use a plain heading and no callout.

Two structural differences worth knowing up front:

- **Diverges from upstream tutorial:** upstream uses a custom "demand / weekday / weekend / price" dataset that it never explains. This notebook uses **California housing** (the same dataset as `c_hyperparameter_tuning.ipynb`) so you can focus on the plot-logging API instead of figuring out the data.  
- **Diverges from upstream tutorial:** upstream shows nine plots (time series, box, density, coefficients, …). This notebook keeps the five most teaching-relevant: a **correlation heatmap** and a **feature-vs-target scatter** as EDA, and **residuals**, **predicted-vs-actual**, **Q–Q** as diagnostics. The pattern is the same for the others — once you've logged one, you've logged them all.

## Prerequisites

**Diverges from upstream tutorial:**  
Upstream says `pip install matplotlib seaborn`. This repo uses **`uv` exclusively** for dependency management — `pip` is explicitly forbidden (see `CLAUDE.md`).  
`matplotlib` and `scipy` are already pulled in transitively; you only need to add `seaborn` for the heatmap:

```bash
uv add seaborn
```

Commit `pyproject.toml` and `uv.lock` together.

### Start the tracking server

**Missing from upstream tutorial:**  
Before running any cell that calls `mlflow.set_tracking_uri(...)`, start the MLflow server **in a separate terminal**, from `src/`:

```bash
cd src/
mlflow ui --host 127.0.0.1 --port 5001
```

Starting from `src/` keeps per-developer runtime state (`mlflow.db`, `mlartifacts/`) inside the source tree — same convention as the previous notebooks. Update `PORT` below if you use a different port.

## Why log plots to MLflow?

The same plot saved in three different places has three different futures:

| Where you saved it | Six weeks later, can you tell which run it came from? |
|---|---|
| Inline in a notebook | Only if the notebook was re-run since — otherwise the figure is from some prior run with prior params. |
| `plots/residuals.png` in your repo | No. There's no link to a `run_id`, params, or metrics; one careless `git add` and the file silently outlives the experiment that produced it. |
| MLflow run artifact | Yes. The figure is bundled with its `run_id`, every param, every metric, the data signature, and the model. |

That last column is what `mlflow.log_figure(fig, "name.png")` buys you. Two more concrete payoffs:

- **The UI renders them inline.** Click a run → Artifacts tab → the PNGs preview without leaving the browser.  
- **They survive a re-run.** Every `start_run()` produces its own artifact directory, so re-running this notebook never overwrites yesterday's plots.

## The plot-function pattern

Each plot below lives inside a small function that **builds the figure and returns it** — it does not call `plt.show()`, and it does not log anything itself. The logging happens once at the end, inside a single `with mlflow.start_run():` block.

Why this split?

- **The function is reusable.** You can call it from a notebook for ad-hoc exploration, from a training script for production logging, or from a test — without changing the function.  
- **The run stays atomic.** Either all five figures land on the same run, or none do. No "oops, the residuals plot went to run A and the Q–Q plot went to run B because I re-ran one cell."  
- **Figures are values, not side effects.** Returning the `Figure` object makes the function testable and easy to compose; `plt.show()`-style code is implicit global state.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.exceptions import RestException

HOST = "127.0.0.1"
PORT = 5001
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

# Idempotent experiment creation — see principle 7 of the
# mlflow-tutorial-improve skill. Re-running this cell is a no-op.
experiment_name = "Logging Plots with MLflow"
try:
    mlflow.create_experiment(name=experiment_name)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(experiment_name)

## Step 1: Load the data

California housing: 8 numeric features (median income, house age, average rooms, …), one regression target (median house value, in $100k). Small enough that Ridge fits in well under a second.

In [ ]:
raw = fetch_california_housing(as_frame=True)
df = raw.frame  # features + target in one DataFrame, handy for the heatmap below
X, y = raw.data, raw.target

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)
df.head(3)

## Step 2: Define the plot functions

Five functions, all returning a `matplotlib.figure.Figure`. Two for EDA (you'd run these *before* picking a model), three for diagnostics (you'd run these *after* fitting one).

In [ ]:
def plot_correlation_heatmap(frame: pd.DataFrame) -> plt.Figure:
    """Heatmap of pairwise Pearson correlations across features + target."""
    fig, ax = plt.subplots(figsize=(8, 6))
    corr = frame.corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # hide upper triangle
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
    ax.set_title("Feature ↔ target correlations")
    fig.tight_layout()
    return fig


def plot_feature_vs_target(frame: pd.DataFrame, feature: str, target: str) -> plt.Figure:
    """Scatter of one feature against the target. Pick the strongest predictor for the most useful plot."""
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(frame[feature], frame[target], s=4, alpha=0.3)
    ax.set_xlabel(feature)
    ax.set_ylabel(target)
    ax.set_title(f"{feature} vs {target}")
    fig.tight_layout()
    return fig


def plot_residuals(y_true: np.ndarray, y_pred: np.ndarray) -> plt.Figure:
    """Residuals vs predicted. Healthy: flat band around 0. Unhealthy: a funnel, a curve, or a tilt."""
    fig, ax = plt.subplots(figsize=(7, 5))
    residuals = y_true - y_pred
    ax.scatter(y_pred, residuals, s=4, alpha=0.3)
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("predicted")
    ax.set_ylabel("residual (true − predicted)")
    ax.set_title("Residuals vs predicted")
    fig.tight_layout()
    return fig


def plot_predicted_vs_actual(y_true: np.ndarray, y_pred: np.ndarray) -> plt.Figure:
    """Predicted vs actual with the y = x reference line. Points off the diagonal are model error."""
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, s=4, alpha=0.3)
    lo, hi = float(min(y_true.min(), y_pred.min())), float(max(y_true.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("actual")
    ax.set_ylabel("predicted")
    ax.set_title("Predicted vs actual")
    fig.tight_layout()
    return fig


def plot_qq(residuals: np.ndarray) -> plt.Figure:
    """Q–Q plot of residuals against a normal distribution. Straight line = residuals are roughly normal."""
    fig, ax = plt.subplots(figsize=(6, 6))
    stats.probplot(residuals, dist="norm", plot=ax)
    ax.set_title("Q–Q plot of residuals (vs normal)")
    fig.tight_layout()
    return fig

## Step 3: Fit a Ridge regression

Ridge is enough to produce a believable residual pattern; this notebook is about logging plots, not about beating a leaderboard.

In [ ]:
params = {"alpha": 1.0, "random_state": 0}
model = Ridge(**params).fit(X_train, y_train)

y_pred = model.predict(X_val)
metrics = {
    "mse": mean_squared_error(y_val, y_pred),
    "r2": r2_score(y_val, y_pred),
}
metrics

## Step 4: Log everything in one run

All five figures, the params, the metrics, and the model land on a single run. `mlflow.log_figure(fig, "name.png")` takes the in-memory `Figure` directly — no need to save to disk first.

**Stale in upstream tutorial:**  
Upstream also shows the `mlflow.log_artifact("/tmp/corr_plot.png")` route, which saves the figure to a temp file and then uploads. That's the older pattern; `mlflow.log_figure(fig, "corr_plot.png")` does both steps in one call and skips the temp file entirely. Stick with `log_figure` unless you're logging something that's already on disk for an unrelated reason.

In [ ]:
# NOTE (differs from upstream tutorial): serialization_format="skops" avoids
# pickle/cloudpickle. See `b_tracking_quickstart.ipynb` for the full rationale.

with mlflow.start_run(run_name="ridge-with-plots"):
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    # EDA plots — properties of the data, not the model.
    mlflow.log_figure(plot_correlation_heatmap(df), "eda/correlation_heatmap.png")
    mlflow.log_figure(plot_feature_vs_target(df, "MedInc", "MedHouseVal"), "eda/medinc_vs_target.png")

    # Diagnostic plots — properties of this particular fit.
    mlflow.log_figure(plot_residuals(y_val.values, y_pred), "diagnostics/residuals.png")
    mlflow.log_figure(plot_predicted_vs_actual(y_val.values, y_pred), "diagnostics/predicted_vs_actual.png")
    mlflow.log_figure(plot_qq(y_val.values - y_pred), "diagnostics/qq.png")

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        input_example=X_train.head(),
        serialization_format="skops",
    )

## Step 5: View the plots in the UI

Open <http://127.0.0.1:5001/>, click into **Logging Plots with MLflow** → the `ridge-with-plots` run → the **Artifacts** tab. The left sidebar shows the directory shape we passed to `log_figure`:

```text
diagnostics/
    predicted_vs_actual.png
    qq.png
    residuals.png
eda/
    correlation_heatmap.png
    medinc_vs_target.png
model/
    MLmodel
    model.skops
    …
```

Two details worth noticing:

- **The `/` in the artifact name became a folder.** Passing `"eda/correlation_heatmap.png"` to `log_figure` creates an `eda/` subdirectory on the artifact store — useful when you log a dozen plots and want them grouped.  
- **The plots live next to the model**, not in a separate place. Anyone who loads this run gets the figures *and* the artifact they describe, in one fetch.

## Next steps

- [MLflow `log_figure` API](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_figure) — supported figure types (matplotlib, plotly), keyword arguments.  
- [Logging plots with MLflow (upstream)](https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/part2-logging-plots/) — the four plot types this notebook skipped (time-series, box, density, coefficients) follow the same pattern.  
- **Combine with `c_hyperparameter_tuning.ipynb`:** log per-trial diagnostic plots to each child run, and log the Optuna study summary (`optuna.visualization.matplotlib.plot_optimization_history(study)`) to the parent run. Same `log_figure` call, different scope.